# Backpropagation 실습

이 노트북에서는 `forward -> loss -> backward -> update` 흐름을 직접 확인합니다.

- `loss.backward()`의 역할 이해
- 각 파라미터의 gradient 확인
- gradient를 이용한 수동 업데이트
- 여러 번 반복했을 때 loss, accuracy 변화 관찰


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 재현성 고정
np.random.seed(42)
torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 120

print("NumPy version  :", np.__version__)
print("PyTorch version:", torch.__version__)


## 1. 작은 식에서 gradient 보기

아주 작은 계산식에서도 PyTorch는 자동으로 계산 그래프를 만들고 미분을 수행합니다.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(-0.4, requires_grad=True)
b = torch.tensor(0.3, requires_grad=True)
t = torch.tensor(1.0)

y = w * x + b
loss = (y - t) ** 2
loss.backward()

print(f"y      = {y.item():.4f}")
print(f"loss   = {loss.item():.4f}")
print(f"dL/dw  = {w.grad.item():.4f}")
print(f"dL/db  = {b.grad.item():.4f}")
print(f"dL/dx  = {x.grad.item():.4f}")


## 2. MNIST 데이터 준비

한 배치를 가져와 이미지와 label을 확인합니다.


In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
images, labels = next(iter(train_loader))

print("images shape:", tuple(images.shape))
print("labels shape:", tuple(labels.shape))
print("first 10 labels:", labels[:10].tolist())

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(images[i, 0], cmap="gray")
    ax.set_title(str(labels[i].item()))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. 간단한 모델과 Loss 정의

이번 실습에서는 `Flatten -> Linear -> ReLU -> Linear` 구조를 사용합니다.


In [ ]:
def build_model(hidden_dim=128):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 10),
    )


model = build_model()
criterion = nn.CrossEntropyLoss()

logits = model(images)
loss = criterion(logits, labels)

print(model)
print("logits shape:", tuple(logits.shape))
print(f"loss = {loss.item():.4f}")


## 4. Backward 수행하기

`loss.backward()`를 호출하면 각 파라미터의 `.grad`에 미분값이 저장됩니다.


In [ ]:
model.zero_grad()
loss.backward()

for name, param in model.named_parameters():
    print(name)
    print("  shape     :", tuple(param.shape))
    print("  grad shape:", tuple(param.grad.shape))
    print("  grad norm :", f"{param.grad.norm().item():.6f}")


## 5. Gradient로 직접 update 하기

optimizer를 쓰지 않고 `param -= lr * grad` 형태로 한 번 직접 업데이트합니다.


In [ ]:
model = build_model()
lr = 0.1

before_loss = criterion(model(images), labels).item()

model.zero_grad()
loss = criterion(model(images), labels)
loss.backward()

with torch.no_grad():
    for param in model.parameters():
        param -= lr * param.grad

after_loss = criterion(model(images), labels).item()

print(f"before update loss = {before_loss:.4f}")
print(f"after update loss  = {after_loss:.4f}")


## 6. 여러 번 반복해보기

같은 배치에 대해 여러 번 update 하면서 loss와 accuracy가 어떻게 바뀌는지 확인합니다.


In [ ]:
model = build_model()
criterion = nn.CrossEntropyLoss()
lr = 0.1

loss_history = []
acc_history = []

for step in range(30):
    logits = model(images)
    loss = criterion(logits, labels)

    model.zero_grad()
    loss.backward()

    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    preds = logits.argmax(dim=1)
    acc = (preds == labels).float().mean().item()
    loss_history.append(loss.item())
    acc_history.append(acc)

print(f"initial loss = {loss_history[0]:.4f}")
print(f"final loss   = {loss_history[-1]:.4f}")
print(f"initial acc  = {acc_history[0]:.4f}")
print(f"final acc    = {acc_history[-1]:.4f}")

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(loss_history, marker="o")
plt.title("Loss over updates")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc_history, marker="o", color="tab:orange")
plt.title("Accuracy over updates")
plt.xlabel("Step")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. 연습

1. `w`, `b`, `x`의 gradient가 각각 무엇을 의미하는지 설명해보세요.
2. learning rate를 `0.01`, `0.5`로 바꿔 loss 감소 속도를 비교해보세요.
3. hidden dimension을 `32`, `256`으로 바꿔 gradient shape와 loss 변화를 비교해보세요.
4. update 반복 횟수를 `30`에서 `100`으로 늘려 accuracy 변화를 확인해보세요.
5. `ReLU`를 `Sigmoid`로 바꿔 loss 감소 패턴이 어떻게 달라지는지 확인해보세요.


## 8. 연습문제 풀이

1. `dL/dw`는 가중치 `w`를 얼마나 바꿔야 loss가 줄어드는지, `dL/db`는 bias에 대한 같은 정보, `dL/dx`는 입력 `x`가 loss에 얼마나 민감한지 보여줍니다.
2. learning rate가 작으면 천천히 줄고, 너무 크면 loss가 크게 흔들릴 수 있습니다.
3. hidden dimension이 커지면 gradient tensor shape와 파라미터 수가 함께 커집니다.
4. 같은 배치를 더 오래 학습시키면 보통 accuracy가 더 올라갑니다.
5. `Sigmoid`는 `ReLU`보다 출력이 눌려서 gradient가 더 작아질 수 있고, loss 감소가 더 느릴 수 있습니다.


In [ ]:
def build_model_with_activation(hidden_dim=128, activation="relu"):
    act_layer = nn.ReLU() if activation == "relu" else nn.Sigmoid()
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, hidden_dim),
        act_layer,
        nn.Linear(hidden_dim, 10),
    )


def run_manual_updates(hidden_dim=128, activation="relu", lr=0.1, steps=30):
    model = build_model_with_activation(hidden_dim=hidden_dim, activation=activation)
    criterion = nn.CrossEntropyLoss()
    loss_history = []
    acc_history = []
    grad_shapes = {}
    grad_norms = {}

    for step in range(steps):
        logits = model(images)
        loss = criterion(logits, labels)

        model.zero_grad()
        loss.backward()

        if step == 0:
            for name, param in model.named_parameters():
                grad_shapes[name] = tuple(param.grad.shape)
                grad_norms[name] = round(param.grad.norm().item(), 6)

        with torch.no_grad():
            for param in model.parameters():
                param -= lr * param.grad

        preds = logits.argmax(dim=1)
        acc = (preds == labels).float().mean().item()
        loss_history.append(loss.item())
        acc_history.append(acc)

    return {
        "loss_history": loss_history,
        "acc_history": acc_history,
        "grad_shapes": grad_shapes,
        "grad_norms": grad_norms,
    }


print("[Exercise 2] learning rate 비교")
for lr in [0.01, 0.1, 0.5]:
    result = run_manual_updates(lr=lr, steps=30)
    print(
        f"lr={lr:>4} | initial_loss={result['loss_history'][0]:.4f} | final_loss={result['loss_history'][-1]:.4f}"
    )


print("\n[Exercise 3] hidden dimension 비교")
for hidden_dim in [32, 128, 256]:
    result = run_manual_updates(hidden_dim=hidden_dim, steps=30)
    print(
        f"hidden_dim={hidden_dim:3d} | fc1.weight grad shape={result['grad_shapes']['1.weight']} | final_loss={result['loss_history'][-1]:.4f}"
    )


In [ ]:
result_30 = run_manual_updates(steps=30)
result_100 = run_manual_updates(steps=100)
result_sigmoid = run_manual_updates(activation="sigmoid", steps=30)

print("[Exercise 4] 반복 횟수 비교")
print(f"30 steps  -> final_acc={result_30['acc_history'][-1]:.4f}")
print(f"100 steps -> final_acc={result_100['acc_history'][-1]:.4f}")

print("\n[Exercise 5] ReLU vs Sigmoid")
print(f"ReLU    final_loss={result_30['loss_history'][-1]:.4f}")
print(f"Sigmoid final_loss={result_sigmoid['loss_history'][-1]:.4f}")
print(f"ReLU    fc1.weight grad norm={result_30['grad_norms']['1.weight']:.6f}")
print(f"Sigmoid fc1.weight grad norm={result_sigmoid['grad_norms']['1.weight']:.6f}")

plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(result_30['loss_history'], label='ReLU (30)')
plt.plot(result_sigmoid['loss_history'], label='Sigmoid (30)')
plt.title('Loss comparison')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(result_30['acc_history'], label='30 steps')
plt.plot(result_100['acc_history'], label='100 steps')
plt.title('Accuracy comparison')
plt.xlabel('Step')
plt.ylabel('Accuracy')
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()
